# 1.0 — Ingestão Forecast Enriched Raw

- **Propósito:** Ingerir os arquivos Excel de forecast enriquecido do volume e persistí-los como tabela Delta.
- **Entrada:** `/Volumes/parts_hdbk_sandbox/pr_forecast/forecast_enriched/current/*.xlsx`
- **Saída:** `parts_hdbk_sandbox.pr_forecast.raw_forecast_enriched`
- **Chave:** `forecast_cycle` + `segment` + `main_material` · **Carga:** Append
- **Auditoria:** `_ingested_at`, `_ingested_by`, `_load_type`, `_load_id`, `_source_file_name`, `_source_file_path`

In [0]:
%pip install openpyxl -q
dbutils.library.restartPython()

In [0]:
import pandas as pd
import os
import uuid
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, TimestampType
)

In [0]:
# ---------------------------------------------------------------------------
# Parâmetros
# ---------------------------------------------------------------------------
VOLUME_PATH = "/Volumes/parts_hdbk_sandbox/pr_forecast/forecast_enriched/current"
CATALOG     = "parts_hdbk_sandbox"
SCHEMA      = "pr_forecast"
TABLE_NAME  = "raw_forecast_enriched"
FULL_TABLE  = f"{CATALOG}.{SCHEMA}.{TABLE_NAME}"

# ---------------------------------------------------------------------------
# Auditoria
# ---------------------------------------------------------------------------
LOAD_MODE    = "append"
LOAD_ID      = str(uuid.uuid4())
CURRENT_USER = spark.sql("SELECT current_user()").first()[0]

print(f"Volume  : {VOLUME_PATH}")
print(f"Tabela  : {FULL_TABLE}")
print(f"Load ID : {LOAD_ID}")
print(f"Usuário : {CURRENT_USER}")

In [0]:
# Verifica se há arquivos .xlsx no volume
files = [f for f in os.listdir(VOLUME_PATH) if f.endswith(".xlsx")]
assert len(files) > 0, f"Nenhum arquivo .xlsx encontrado em {VOLUME_PATH}"
print(f"Arquivos encontrados ({len(files)}):")
for f in sorted(files):
    print(f"  • {f}")

In [0]:
# Lê todos os .xlsx como string para preservar zeros à esquerda em todas as colunas
frames = []
for f in sorted(files):
    path = os.path.join(VOLUME_PATH, f)
    pdf = pd.read_excel(path, dtype=str)
    pdf["_source_file_name"] = f
    frames.append(pdf)
    print(f"  {f}: {len(pdf):,} linhas")

pdf_all = pd.concat(frames, ignore_index=True)
print(f"\nTotal: {len(pdf_all):,} linhas")

# ---------------------------------------------------------------------------
# Mapeamento de tipos de dados
#   forecast_cycle  -> DateType
#   n+0 .. n+35     -> DoubleType  (valores com casas decimais)
#   demais colunas  -> StringType
# ---------------------------------------------------------------------------
HORIZON_COLS = {f"n+{i}" for i in range(36)}

COLUMN_TYPE_MAP = {}
for col in pdf_all.columns:
    if col == "forecast_cycle":
        COLUMN_TYPE_MAP[col] = "date"
    elif col in HORIZON_COLS:
        COLUMN_TYPE_MAP[col] = "double"
    else:
        COLUMN_TYPE_MAP[col] = "string"

print("\nMapa de tipos de dados:")
for col, dtype in COLUMN_TYPE_MAP.items():
    print(f"  {col:<28} -> {dtype}")

# Converte para Spark DataFrame e aplica tipagem com withColumns (evita loop)
df = spark.createDataFrame(pdf_all)
df = df.withColumns({col: F.col(f"`{col}`").cast(dtype) for col, dtype in COLUMN_TYPE_MAP.items()})

# ---------------------------------------------------------------------------
# Colunas de auditoria (padrão do projeto Honda Demand)
# ---------------------------------------------------------------------------
df = df.withColumns({
    "_source_file_path": F.concat(
        F.lit(VOLUME_PATH + "/"), F.col("_source_file_name")
    ),
    "_ingested_at": F.from_utc_timestamp(
        F.current_timestamp(), "America/Sao_Paulo"
    ),
    "_ingested_by": F.lit(CURRENT_USER),
    "_load_type": F.lit(LOAD_MODE),
    "_load_id": F.lit(LOAD_ID),
})

print(f"\nSchema Spark:")
df.printSchema()

In [0]:
display(df.limit(10))

In [0]:
# Persiste em modo append na tabela Delta
df.write.mode("append").saveAsTable(FULL_TABLE)
print(f"Dados inseridos com sucesso em {FULL_TABLE} (modo append)")

In [0]:
# Validação resumida da tabela após inserção
df_final = spark.table(FULL_TABLE)
print(f"Total de linhas na tabela: {df_final.count():,}")

print(f"\nDistribuição por segment:")
display(
    df_final.groupBy("segment")
    .agg(F.count("*").alias("qtd_linhas"))
    .orderBy("segment")
)

print(f"\nAuditoria da última carga:")
display(
    df_final
    .groupBy("_load_id", "_ingested_by", "_load_type")
    .agg(
        F.count("*").alias("qtd_linhas"),
        F.min("_ingested_at").alias("ingested_at"),
        F.collect_set("_source_file_name").alias("arquivos"),
    )
    .orderBy(F.col("ingested_at").desc())
)